# VCR-Bench Starting Guide

This notebook shows two ways to use the project:

- notebook shell-style runs through the same CLI entry points
- the Python API directly

For each style, it runs:

1. a clean test
2. an `ifgsm` attack
3. the same attack with `gaussian_blur`

It also installs the optional metric dependencies, configures an FFmpeg build with `libvmaf`, and includes a reusable attack/defence matrix cell for other combinations.

Dataset subsets and checkpoints are resolved automatically when missing.


In [ ]:
import json, os, platform, shutil, signal, subprocess, sys, tarfile, urllib.request, zipfile
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = next((p for p in [ROOT, *ROOT.parents] if (p / "pyproject.toml").exists()), ROOT)

OUT = ROOT / "notebook_outputs"
OUT.mkdir(exist_ok=True)

def run_cmd(args, **kwargs):
    print("$", " ".join(map(str, args)))
    result = subprocess.run(list(map(str, args)), check=False, **kwargs)
    sigkill = getattr(signal, "SIGKILL", None)
    if sigkill is not None and result.returncode == -sigkill:
        print("Command was killed with SIGKILL. This usually means the OS or GPU driver killed it for RAM/VRAM pressure.")
        print("Try NOTEBOOK_RUN_PROFILE='smoke', lower NUM_VIDEOS, or disable CALC_LPIPS/CALC_VMAF for the first pass.")
    result.check_returncode()
    return result


def run_json(args, output_path: Path):
    run_cmd(args)
    return json.loads(output_path.read_text(encoding="utf-8"))


## Install VCR-Bench And Metric Extras


In [ ]:
%cd {ROOT}
run_cmd([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])

# Auto-install a CUDA PyTorch wheel when an NVIDIA GPU is visible.
# The wheel includes CUDA runtime libraries; the NVIDIA driver still must exist on the host.
INSTALL_TORCH_CUDA = "auto"  # "auto", True, or False
TORCH_CUDA_INDEX = "https://download.pytorch.org/whl/cu128"


def nvidia_gpu_visible():
    if shutil.which("nvidia-smi") is None:
        return False
    try:
        result = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=10)
        return result.returncode == 0 and "GPU" in result.stdout
    except Exception:
        return False


if INSTALL_TORCH_CUDA == "auto":
    install_torch_cuda = nvidia_gpu_visible() and platform.system().lower() in {"linux", "windows"}
else:
    install_torch_cuda = bool(INSTALL_TORCH_CUDA)

if install_torch_cuda:
    run_cmd([
        sys.executable, "-m", "pip", "install",
        "torch", "torchvision",
        "--index-url", TORCH_CUDA_INDEX,
    ])
else:
    print("CUDA PyTorch preinstall skipped; project install will resolve torch from the default pip index.")

# Install the project and notebook display dependencies first. Keep this separate from
# research extras so torch/pandas and plotting support are available even if an optional metric
# package is temporarily unavailable for your Python version.
run_cmd([sys.executable, "-m", "pip", "install", "-e", ".", "matplotlib", "pickleshare"])

# Optional metric/research dependencies used by LPIPS/DISTS and some heavier attacks.
# IQA-pytorch currently publishes only version 0.1 on PyPI, so the project metadata pins
# the research extra to the installable range.
INSTALL_RESEARCH_EXTRAS = True
RESEARCH_EXTRAS_AVAILABLE = False
if INSTALL_RESEARCH_EXTRAS:
    try:
        run_cmd([sys.executable, "-m", "pip", "install", "-e", ".[research]"])
        RESEARCH_EXTRAS_AVAILABLE = True
    except subprocess.CalledProcessError as exc:
        print(f"Research extras failed to install (exit {exc.returncode}).")
        print("Continuing without LPIPS/DISTS metric extras; the core notebook dependencies are installed.")

import torch
import matplotlib
import pandas as pd
print("torch:", torch.__version__)
print("torch cuda build:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("matplotlib:", matplotlib.__version__)
print("pandas:", pd.__version__)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

if RESEARCH_EXTRAS_AVAILABLE:
    # LPIPS is provided by the research extra via IQA-pytorch. Fail early here
    # instead of discovering missing LPIPS after a long attack run.
    from IQA_pytorch import LPIPSvgg
    print("LPIPS dependency is available:", LPIPSvgg.__name__)
else:
    print("Research extras are unavailable. LPIPS/DISTS metrics will be disabled below.")


## Install FFmpeg With libvmaf

Most system FFmpeg packages are built without the `libvmaf` filter. This cell downloads a ready GPL build from BtbN FFmpeg-Builds, points VCR-Bench to it through `FFMPEG_BIN`, and checks that `libvmaf` is actually present before VMAF is enabled.


In [ ]:
FFMPEG_DIR = ROOT / "notebook_tools" / "ffmpeg"
FFMPEG_DIR.mkdir(parents=True, exist_ok=True)

system = platform.system().lower()
machine = platform.machine().lower()
if system == "linux" and machine in {"x86_64", "amd64"}:
    ffmpeg_url = "https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-linux64-gpl.tar.xz"
    archive_path = FFMPEG_DIR / "ffmpeg-linux64-gpl.tar.xz"
elif system == "windows" and machine in {"amd64", "x86_64"}:
    ffmpeg_url = "https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-win64-gpl.zip"
    archive_path = FFMPEG_DIR / "ffmpeg-win64-gpl.zip"
else:
    ffmpeg_url = None
    archive_path = None

if ffmpeg_url is None:
    raise RuntimeError(f"No bundled FFmpeg/libvmaf recipe for {platform.system()} {platform.machine()}. Install FFmpeg with libvmaf and set FFMPEG_BIN manually.")

ffmpeg_name = "ffmpeg.exe" if system == "windows" else "ffmpeg"
existing = sorted(FFMPEG_DIR.rglob(ffmpeg_name))
ffmpeg_bin = existing[0] if existing else None

if ffmpeg_bin is None:
    print("Downloading", ffmpeg_url)
    urllib.request.urlretrieve(ffmpeg_url, archive_path)
    if archive_path.suffix == ".zip":
        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall(FFMPEG_DIR)
    else:
        with tarfile.open(archive_path) as tf:
            tf.extractall(FFMPEG_DIR)
    ffmpeg_bin = sorted(FFMPEG_DIR.rglob(ffmpeg_name))[0]

if system != "windows":
    ffmpeg_bin.chmod(ffmpeg_bin.stat().st_mode | 0o111)

filters = subprocess.run([str(ffmpeg_bin), "-hide_banner", "-filters"], check=True, capture_output=True, text=True).stdout
if " libvmaf " not in filters:
    raise RuntimeError(f"Downloaded FFmpeg does not expose libvmaf: {ffmpeg_bin}")

os.environ["FFMPEG_BIN"] = str(ffmpeg_bin)
os.environ["VMAF_BACKEND"] = "ffmpeg"
os.environ.setdefault("VMAF_TIMEOUT_SEC", "180")
print("Using FFmpeg:", ffmpeg_bin)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch

# Keep the default notebook profile small enough for a clean-machine smoke run.
# Use NOTEBOOK_RUN_PROFILE="full" for the 8-video walkthrough once smoke passes.
NOTEBOOK_RUN_PROFILE = os.getenv("NOTEBOOK_RUN_PROFILE", "smoke").strip().lower()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# X3D-M: small but accurate K400 model (~3.8M params, ~76% top-1) whose weights are
# Apache-2.0 and auto-downloaded. Swap in any preset from configs/models/ to try others.
MODEL_PRESET = "x3d"
MODEL_PRESET_NAME = "m_k400"
ATTACK_PRESET = "ifgsm"
DEFENCE_PRESET = "gaussian_blur"
DATASET = "kinetics400"
# `demo`: small, redistributable CC BY 3.0 Kinetics subset (real clips + real labels),
# auto-downloaded from Hugging Face. To run on full Kinetics, set DATASET_SUBSET to
# "k400_val"/"k400_test" and supply your own videos via --video-root (see docs/datasets.md).
DATASET_SUBSET = "demo"
NUM_VIDEOS = 8 if NOTEBOOK_RUN_PROFILE == "full" else 1

# Keep both metrics explicit so CLI and Python API runs behave the same way.
# Set CALC_VMAF=False when you need a faster smoke run.
CALC_LPIPS = bool(globals().get("RESEARCH_EXTRAS_AVAILABLE", False))
CALC_VMAF = True
METRIC_WORKERS = os.getenv("VCR_BENCH_METRIC_WORKERS", "auto")  # "auto" or a positive integer

print("notebook profile:", NOTEBOOK_RUN_PROFILE)
print("num videos:", NUM_VIDEOS)
print("metric workers:", METRIC_WORKERS)

## Clean Test

In [ ]:
accuracy_cli_json = OUT / "accuracy_cli.json"
accuracy_cli = run_json([
    sys.executable, "-m", "vcr_bench.cli.test",
    "--model-preset", MODEL_PRESET,
    "--model-preset-name", MODEL_PRESET_NAME,
    "--dataset", DATASET,
    "--dataset-subset", DATASET_SUBSET,
    "--num-videos", NUM_VIDEOS,
    "--device", DEVICE,
    "--output-json", accuracy_cli_json,
], accuracy_cli_json)
accuracy_cli


## Clean Test With Python API

In [ ]:
from vcr_bench.attacks import create_attack
from vcr_bench.datasets import create_dataset
from vcr_bench.defences import create_defence
from vcr_bench.models import create_model
from vcr_bench.presets import resolve_entity_preset
from vcr_bench.utils.eval import run_accuracy, run_attack

model_spec = resolve_entity_preset("model", MODEL_PRESET, preset_name=MODEL_PRESET_NAME)
attack_spec = resolve_entity_preset("attack", ATTACK_PRESET)
defence_spec = resolve_entity_preset("defence", DEFENCE_PRESET)


def make_model():
    params = dict(model_spec["params"])
    params["device"] = DEVICE
    return create_model(model_spec["factory_name"], **params)


def make_dataset():
    return create_dataset(DATASET, dataset_subset=DATASET_SUBSET, split="val")

accuracy_api = run_accuracy(model=make_model(), dataset=make_dataset(), num_videos=NUM_VIDEOS, pipeline_stage="test")
accuracy_api


## IFGSM Attack With CLI


In [ ]:
attack_cli_json = OUT / "attack_cli.json"
attack_cli_args = [
    sys.executable, "-m", "vcr_bench.cli.attack",
    "--model-preset", MODEL_PRESET,
    "--lite-attack",
    "--model-preset-name", MODEL_PRESET_NAME,
    "--attack-preset", ATTACK_PRESET,
    "--dataset", DATASET,
    "--dataset-subset", DATASET_SUBSET,
    "--num-videos", NUM_VIDEOS,
    "--device", DEVICE,
    "--output-json", attack_cli_json,
    "--lpips" if CALC_LPIPS else "--no-lpips",
    "--vmaf" if CALC_VMAF else "--no-vmaf",
    "--metric-workers", METRIC_WORKERS,
]
attack_cli = run_json(attack_cli_args, attack_cli_json)
attack_cli


## IFGSM Attack With Python API

In [ ]:
attack_api = run_attack(
    model=make_model(),
    attack=create_attack(attack_spec["factory_name"], **attack_spec["params"]),
    dataset=make_dataset(),
    save_path=OUT / "attack_api.csv",
    log_path=None,
    attack_name="ifgsm_api",
    num_videos=NUM_VIDEOS,
    pipeline_stage="attack",
    skip_existing=False,
    calc_lpips=CALC_LPIPS,
    calc_vmaf=CALC_VMAF,
    metric_workers=METRIC_WORKERS,
)
attack_api


## IFGSM Attack + Gaussian Blur Defence With CLI


In [ ]:
defence_cli_json = OUT / "defence_cli.json"
defence_cli_args = [
    sys.executable, "-m", "vcr_bench.cli.attack",
    "--model-preset", MODEL_PRESET,
    "--lite-attack",
    "--model-preset-name", MODEL_PRESET_NAME,
    "--attack-preset", ATTACK_PRESET,
    "--defence-preset", DEFENCE_PRESET,
    "--dataset", DATASET,
    "--dataset-subset", DATASET_SUBSET,
    "--num-videos", NUM_VIDEOS,
    "--device", DEVICE,
    "--output-json", defence_cli_json,
    "--lpips" if CALC_LPIPS else "--no-lpips",
    "--vmaf" if CALC_VMAF else "--no-vmaf",
    "--metric-workers", METRIC_WORKERS,
]
defence_cli = run_json(defence_cli_args, defence_cli_json)
defence_cli


## IFGSM + Gaussian Blur With Python API

In [ ]:
defence_api = run_attack(
    model=make_model(),
    attack=create_attack(attack_spec["factory_name"], **attack_spec["params"]),
    dataset=make_dataset(),
    save_path=OUT / "defence_api.csv",
    log_path=None,
    attack_name="ifgsm_blur_api",
    num_videos=NUM_VIDEOS,
    pipeline_stage="attack",
    defence=create_defence(defence_spec["factory_name"], **defence_spec["params"]),
    skip_existing=False,
    calc_lpips=CALC_LPIPS,
    calc_vmaf=CALC_VMAF,
    metric_workers=METRIC_WORKERS,
)
defence_api


## Compare CLI And Python API Results

In [ ]:
def attack_success_rate(summary):
    denom = max(int(summary.get("clear_correct", 0)), 1)
    return float(summary.get("attacked_success", 0)) / denom

comparison = pd.DataFrame([
    {"source": "cli", "run": "clean", "accuracy": float(accuracy_cli.get("accuracy", 0.0))},
    {"source": "api", "run": "clean", "accuracy": float(accuracy_api.get("accuracy", 0.0))},
    {"source": "cli", "run": "ifgsm", "attack_success_rate": attack_success_rate(attack_cli), "psnr": float(attack_cli.get("mean_psnr", 0.0)), "vmaf": float(attack_cli.get("mean_vmaf", 0.0))},
    {"source": "api", "run": "ifgsm", "attack_success_rate": attack_success_rate(attack_api), "psnr": float(attack_api.get("mean_psnr", 0.0)), "vmaf": float(attack_api.get("mean_vmaf", 0.0))},
    {"source": "cli", "run": "ifgsm+blur", "attack_success_rate": attack_success_rate(defence_cli), "psnr": float(defence_cli.get("mean_psnr", 0.0)), "vmaf": float(defence_cli.get("mean_vmaf", 0.0))},
    {"source": "api", "run": "ifgsm+blur", "attack_success_rate": attack_success_rate(defence_api), "psnr": float(defence_api.get("mean_psnr", 0.0)), "vmaf": float(defence_api.get("mean_vmaf", 0.0))},
])
comparison


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(19, 4))

clean = comparison[comparison.run == "clean"]
axes[0].bar(clean["source"], clean["accuracy"].fillna(0))
axes[0].set_title("Clean Accuracy")
axes[0].set_ylim(0, 1)

attack_runs = comparison[comparison["attack_success_rate"].notna()]
labels = attack_runs["source"] + "-" + attack_runs["run"]
axes[1].bar(labels, attack_runs["attack_success_rate"])
axes[1].set_title("Attack Success Rate")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=30)

axes[2].bar(labels, attack_runs["psnr"])
axes[2].set_title("Mean PSNR")
axes[2].tick_params(axis="x", rotation=30)

axes[3].bar(labels, attack_runs["vmaf"])
axes[3].set_title("Mean VMAF")
axes[3].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


## Inspect Per-Video Rows

The CLI and Python API attack summaries include `save_path`, so you can open the generated CSV files the same way.


In [ ]:
pd.read_csv(attack_cli["save_path"])[["video_name", "clear_class", "attacked_class", "psnr", "lpips", "vmaf", "time"]].head()


## Attack/Defence Matrix

Use this helper to run other attack/defence combinations without editing the earlier cells. Keep `RUN_COMBINATION_GRID = False` for a fast full-notebook smoke run; switch it to `True` when you want to execute the grid.


In [ ]:
ATTACK_GRID = ["ifgsm", "mifgsm", "square"]
DEFENCE_GRID = [None, "gaussian_blur", "jpeg_compression", "temporal_median"]
GRID_NUM_VIDEOS = 2 if NOTEBOOK_RUN_PROFILE == "full" else 1
RUN_COMBINATION_GRID = False


def run_attack_defence_combo(attack_preset, defence_preset=None, *, num_videos=GRID_NUM_VIDEOS):
    attack_name = attack_preset if defence_preset is None else f"{attack_preset}+{defence_preset}"
    output_json = OUT / f"combo_{attack_name.replace('+', '_')}.json"
    args = [
        sys.executable, "-m", "vcr_bench.cli.attack",
        "--model-preset", MODEL_PRESET,
        "--lite-attack",
        "--model-preset-name", MODEL_PRESET_NAME,
        "--attack-preset", attack_preset,
        "--dataset", DATASET,
        "--dataset-subset", DATASET_SUBSET,
        "--num-videos", num_videos,
        "--device", DEVICE,
        "--output-json", output_json,
        "--lpips" if CALC_LPIPS else "--no-lpips",
        "--vmaf" if CALC_VMAF else "--no-vmaf",
        "--metric-workers", METRIC_WORKERS,
    ]
    if defence_preset is not None:
        args += ["--defence-preset", defence_preset]
    result = run_json(args, output_json)
    result["combo"] = attack_name
    return result

combo_plan = pd.DataFrame(
    {"attack": attack, "defence": defence or "none"}
    for attack in ATTACK_GRID
    for defence in DEFENCE_GRID
)
combo_plan


In [ ]:
if RUN_COMBINATION_GRID:
    combo_results = [
        run_attack_defence_combo(attack, defence)
        for attack in ATTACK_GRID
        for defence in DEFENCE_GRID
    ]
    combo_results = pd.DataFrame(combo_results)
else:
    combo_results = pd.DataFrame(columns=["combo", "processed", "clear_correct", "attacked_success", "mean_psnr", "mean_vmaf"])
combo_results


## Next Steps

- Change `MODEL_PRESET` and `MODEL_PRESET_NAME` to try another model.
- Replace `ATTACK_PRESET` with another attack preset.
- Replace `DEFENCE_PRESET` with another defence preset.
- Add presets to `ATTACK_GRID` and `DEFENCE_GRID`, then set `RUN_COMBINATION_GRID = True` to execute a small matrix.
- Inspect attack-specific flags with `python -m vcr_bench.cli.attack --attack square --print-attack-spec`.
